# 🏦 JPMC Consumer Banking - Hands-on Lab
## Notebook 08a: Spark on SPCS - Preparation & Setup

---

### What This Notebook Does
- Creates **External Access Integration** to allow SPCS containers to reach Maven/PyPI
- Creates the **Compute Pool** for running Spark on SPCS
- Creates the **Notebook object** (08b) that runs Spark with Iceberg + Horizon Catalog

### Prerequisites
- ACCOUNTADMIN role
- Notebook 02 completed (Iceberg tables + Catalog Integration exist)
- Git repo stage `@HOL_GIT_REPO` configured

> **Runtime:** Warehouse-backed (standard notebook)
> **Warehouse:** `HOL_MEDIUM_WH`

In [ ]:
-- =============================================================================
-- SETUP CONTEXT
-- =============================================================================

USE ROLE ACCOUNTADMIN;
USE DATABASE JPMC_CONSUMER_BANKING_HOL;
USE SCHEMA HOL_LAB;
USE WAREHOUSE HOL_MEDIUM_WH;

## Step 1: Create External Access Integration

SPCS containers have no outbound internet by default. We need to allow access to Maven Central (for Iceberg JARs), PyPI (for pip packages), and Ubuntu repos (for Java).

In [ ]:
-- =============================================================================
-- EXTERNAL ACCESS INTEGRATION (allows SPCS container to reach Maven/PyPI)
-- Required so Spark can download Iceberg JARs from repo1.maven.org
-- =============================================================================

-- Step 1: Create a network rule to allow outbound HTTPS to Maven & PyPI
CREATE OR REPLACE NETWORK RULE HOL_MAVEN_NETWORK_RULE
    MODE = EGRESS
    TYPE = HOST_PORT
    VALUE_LIST = ('repo1.maven.org:443', 'repos.spark-packages.org:443',
                  'pypi.org:443', 'files.pythonhosted.org:443',
                  'archive.ubuntu.com:80', 'security.ubuntu.com:80');

-- Step 2: Create the external access integration
CREATE OR REPLACE EXTERNAL ACCESS INTEGRATION HOL_MAVEN_EAI
    ALLOWED_NETWORK_RULES = (HOL_MAVEN_NETWORK_RULE)
    ENABLED = TRUE
    COMMENT = 'Allows SPCS notebooks to download Maven JARs and pip packages';

-- Step 3: Grant usage to SYSADMIN
GRANT USAGE ON INTEGRATION HOL_MAVEN_EAI TO ROLE SYSADMIN;

DESCRIBE INTEGRATION HOL_MAVEN_EAI;

## Step 2: Create Compute Pool for Spark

In [ ]:
-- =============================================================================
-- COMPUTE POOL for Spark on SPCS
-- =============================================================================

CREATE COMPUTE POOL IF NOT EXISTS HOL_SPARK_POOL
    MIN_NODES = 1
    MAX_NODES = 2
    INSTANCE_FAMILY = CPU_X64_M
    AUTO_SUSPEND_SECS = 300
    COMMENT = 'Compute pool for Spark on SPCS - HOL Notebook 08b';

-- Verify compute pool status
SHOW COMPUTE POOLS LIKE 'HOL_SPARK%';

-- Verify catalog integration exists (created in Notebook 02)
SHOW CATALOG INTEGRATIONS LIKE 'HOL_ICEBERG%';

## Step 3: Create the Spark Notebook Object (08b)

This creates the SPCS-backed notebook that will run Spark with Iceberg + Horizon Catalog.
The notebook file comes from the Git repo stage and runs on the compute pool (not a warehouse).

In [ ]:
-- =============================================================================
-- CREATE NOTEBOOK OBJECT (runs on SPCS, not warehouse)
-- =============================================================================

CREATE OR REPLACE NOTEBOOK JPMC_CONSUMER_BANKING_HOL.HOL_LAB."08b_Spark_Iceberg_Horizon"
    FROM '@HOL_GIT_REPO/branches/main/'
    MAIN_FILE = '08b_Spark_Iceberg_Horizon.ipynb'
    COMPUTE_POOL = HOL_SPARK_POOL
    EXTERNAL_ACCESS_INTEGRATIONS = ('HOL_MAVEN_EAI');

-- Add live version
ALTER NOTEBOOK JPMC_CONSUMER_BANKING_HOL.HOL_LAB."08b_Spark_Iceberg_Horizon" ADD LIVE VERSION FROM LAST;

## Step 4: Verify Setup

In [ ]:
-- =============================================================================
-- VERIFY: All resources are ready
-- =============================================================================

-- Check compute pool is ACTIVE or IDLE
SELECT SYSTEM$GET_COMPUTE_POOL_STATUS('HOL_SPARK_POOL');

-- Check notebook exists
SHOW NOTEBOOKS LIKE '08b_Spark%' IN SCHEMA HOL_LAB;

-- Check external access integration
SHOW EXTERNAL ACCESS INTEGRATIONS LIKE 'HOL_MAVEN%';

## Done!

All prerequisites are in place. Now open **Notebook 08b** (`08b_Spark_Iceberg_Horizon`) to:
1. Install PySpark + Java
2. Configure Spark with Snowflake Horizon REST Catalog
3. Query Iceberg tables directly with Spark (no warehouse!)
4. Write results back to Iceberg for Snowflake to read

> The notebook runs on SPCS compute pool `HOL_SPARK_POOL` with `HOL_MAVEN_EAI` attached for internet access.